# Explanation-Guided Reconstruction Experiment

For each HumanEvalPlus or MBPP+ task, the selected coding agent first explains the original code (`C → E`). That session is ended before a fresh session reconstructs code (`C'`) from the explanation alone. The behavioral assessment then executes each available test independently and records the passed count, total count, and pass rate. Structural assessment adds graph-based comparison reports.

Benchmark originals are cached in `data/original_code/{humaneval,mbpp}/`; you can place an edited `.py` file there to override the benchmark source. Generated code, explanations, per-test details, and traces go under `artifacts/`; aggregate CSVs go under `results/`.

In [1]:
from pathlib import Path
import sys

PROJECT_ROOT = Path.cwd().resolve()
if not (PROJECT_ROOT / "experiment.py").is_file():
    raise FileNotFoundError("Open this notebook from the Implementation project root.")
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

import experiment
experiment.describe()

{
  "project_root": "/Users/jj/Desktop/SeniorProject/Implementation",
  "data_root": "/Users/jj/Desktop/SeniorProject/Implementation/data",
  "artifact_root": "/Users/jj/Desktop/SeniorProject/Implementation/artifacts",
  "result_root": "/Users/jj/Desktop/SeniorProject/Implementation/results",
  "datasets": {
    "humaneval": "evalplus/humanevalplus",
    "mbpp": "evalplus/mbppplus"
  },
  "agents": {
    "claude_code": {
      "opus": "claude-opus-5.5",
      "sonnet": "claude-sonnet-5.5",
      "haiku": "claude-haiku-5.5"
    },
    "gpt_codex": {
      "astra": "gpt-6-astra",
      "sol": "gpt-6-sol",
      "luna": "gpt-6-luna"
    },
    "ollama_qwen": {
      "qwen2.5-coder-3b": "qwen2.5-coder:3b",
      "qwen3.5-9b": "qwen3.5:9b",
      "qwen2.5-coder-14b": "qwen2.5-coder:14b"
    },
    "mock": {
      "offline": "mock"
    }
  }
}


{'project_root': '/Users/jj/Desktop/SeniorProject/Implementation',
 'data_root': '/Users/jj/Desktop/SeniorProject/Implementation/data',
 'artifact_root': '/Users/jj/Desktop/SeniorProject/Implementation/artifacts',
 'result_root': '/Users/jj/Desktop/SeniorProject/Implementation/results',
 'datasets': {'humaneval': 'evalplus/humanevalplus',
  'mbpp': 'evalplus/mbppplus'},
 'agents': {'claude_code': {'opus': 'claude-opus-5.5',
   'sonnet': 'claude-sonnet-5.5',
   'haiku': 'claude-haiku-5.5'},
  'gpt_codex': {'astra': 'gpt-6-astra',
   'sol': 'gpt-6-sol',
   'luna': 'gpt-6-luna'},
  'ollama_qwen': {'qwen2.5-coder-3b': 'qwen2.5-coder:3b',
   'qwen3.5-9b': 'qwen3.5:9b',
   'qwen2.5-coder-14b': 'qwen2.5-coder:14b'},
  'mock': {'offline': 'mock'}}}

## Select an agent, model, and benchmark

Change `AGENT` to `claude_code`, `gpt_codex`, or `ollama_qwen` and select a model key listed by `experiment.describe()`. `BATCH_SIZE=3` runs the next three unfinished tasks per dataset; rerunning the cell with the same `RUN_NAME` continues with the next unfinished batch. Keep `RUN_NAME` blank to continue the existing results, or set a name such as `trial-02` to save a separate run under its own artifacts and results folders. Reuse that exact name to resume it. After a run, open `results/index.html` for a clickable overview of all saved results.

### Download/cache originals before calling a model

Run this cell separately to populate `data/original_code/`. The first Hugging Face download may take a little while; it does not call an LLM.

In [2]:
prepared_data = experiment.prepare_dataset(("humaneval", "mbpp"))
prepared_data

Loading humaneval benchmark; this may download it once...


Saved 164 original-code files and loaded 124253 EvalPlus input cases (1570 base, 122683 Plus) to /Users/jj/Desktop/SeniorProject/Implementation/data/original_code/humaneval
Loading mbpp benchmark; this may download it once...
Saved 378 original-code files and loaded 41015 EvalPlus input cases (1174 base, 39841 Plus) to /Users/jj/Desktop/SeniorProject/Implementation/data/original_code/mbpp


{'humaneval': {'tasks': 164,
  'base_test_cases': 1570,
  'plus_test_cases': 122683,
  'original_code_folder': '/Users/jj/Desktop/SeniorProject/Implementation/data/original_code/humaneval',
  'sample_files': ['/Users/jj/Desktop/SeniorProject/Implementation/data/original_code/humaneval/HumanEval_0.py',
   '/Users/jj/Desktop/SeniorProject/Implementation/data/original_code/humaneval/HumanEval_1.py',
   '/Users/jj/Desktop/SeniorProject/Implementation/data/original_code/humaneval/HumanEval_2.py',
   '/Users/jj/Desktop/SeniorProject/Implementation/data/original_code/humaneval/HumanEval_3.py',
   '/Users/jj/Desktop/SeniorProject/Implementation/data/original_code/humaneval/HumanEval_4.py']},
 'mbpp': {'tasks': 378,
  'base_test_cases': 1174,
  'plus_test_cases': 39841,
  'original_code_folder': '/Users/jj/Desktop/SeniorProject/Implementation/data/original_code/mbpp',
  'sample_files': ['/Users/jj/Desktop/SeniorProject/Implementation/data/original_code/mbpp/MBPP_2.py',
   '/Users/jj/Desktop/Sen

In [2]:
AGENT = "ollama_qwen"
MODEL_KEY = "qwen3.5-9b"
DATASETS = ("humaneval", "mbpp")
BATCH_SIZE = None
RUN_NAME = "baseline"  # Keep blank to continue existing results; name new runs, e.g. "trial-02".

# Reload the runner so notebook kernels pick up source changes between runs.
import importlib
experiment = importlib.reload(experiment)

run_output = experiment.run(
    AGENT,
    model_key=MODEL_KEY,
    datasets=DATASETS,
    batch_size=BATCH_SIZE,
    run_name=RUN_NAME,
)
run_output


ollama_qwen / qwen3.5:9b / baseline / humaneval: 164 task(s) to run; 0 resumed
Loaded 164 task(s); original code saved under /Users/jj/Desktop/SeniorProject/Implementation/data/original_code/humaneval
HumanEval/0: starting explanation/reconstruction
HumanEval/0: asking ollama_qwen to explain original code
Waiting for Ollama response from qwen3.5:9b (timeout: 900s)...
................................................................................................................................................................................................................................................................................................................. done
[HumanEval/0] explain: 52.3s
HumanEval/0: asking ollama_qwen to reconstruct from explanation
Waiting for Ollama response from qwen3.5:9b (timeout: 900s)...
 done
[HumanEval/0] reconstruct: 98.7s
HumanEval/0: 0/1006 tests passed
Original source: /Users/jj/Desktop/SeniorProject/Implementation/data/original_code/humanev

KeyboardInterrupt: 

## Configured model keys

- Claude Code: `opus`, `sonnet`, `haiku`
- GPT Codex: `astra`, `sol`, `luna`
- Ollama Qwen: `qwen2.5-coder-3b`, `qwen3.5-9b`, `qwen2.5-coder-14b`

The IDs used by each CLI/server are editable in `config.py` or can be supplied via the documented environment variables. `experiment.run_all(...)` runs all configured model keys and can be expensive; this notebook does not invoke it automatically.

In [ ]:
comparison = experiment.summary_table()
comparison